# Managing Team State - Saving and Loading Teams

In [1]:
import asyncio
from autogen_core import CancellationToken
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
model_client = OpenAIChatCompletionClient(model='gpt-4o', api_key=api_key)

In [2]:
from autogen_agentchat.conditions import MaxMessageTermination

agent_1 = AssistantAgent(
    name='Writer_1',
    model_client=model_client,
    system_message="You are a helpful assistant. Give the output in less than 30 words",
)

agent_2 = AssistantAgent(
    name='Writer_2',
    model_client=model_client,
    system_message="You are a helpful assistant.Give the output in less than 30 words",
)

terminationCondition = MaxMessageTermination(max_messages=3)

agent_team = RoundRobinGroupChat(participants=[agent_1, agent_2],termination_condition=terminationCondition)

In [3]:
from autogen_agentchat.ui import Console

stream = agent_team.run_stream(task="Write a poem about the sea in 3 lines")

await Console(stream)

---------- TextMessage (user) ----------
Write a poem about the sea in 3 lines
---------- TextMessage (Writer_1) ----------
Vast waves dance with grace,  
whispering secrets untold,  
blue embrace of dreams.
---------- TextMessage (Writer_2) ----------
Endless blue whispers,  
moonlit waves cradle the stars,  
secrets in the deep.


TaskResult(messages=[TextMessage(id='88af68a4-bd6d-4b27-b2c0-a23e00a64db4', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 59, 6, 442141, tzinfo=datetime.timezone.utc), content='Write a poem about the sea in 3 lines', type='TextMessage'), TextMessage(id='79d285d3-1511-4b5e-b132-47221331ed7f', source='Writer_1', models_usage=RequestUsage(prompt_tokens=37, completion_tokens=21), metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 59, 8, 431329, tzinfo=datetime.timezone.utc), content='Vast waves dance with grace,  \nwhispering secrets untold,  \nblue embrace of dreams.', type='TextMessage'), TextMessage(id='546ecd78-3e3b-4177-8324-572abe4e29a9', source='Writer_2', models_usage=RequestUsage(prompt_tokens=66, completion_tokens=20), metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 59, 9, 256680, tzinfo=datetime.timezone.utc), content='Endless blue whispers,  \nmoonlit waves cradle the stars,  \nsecrets in the deep.', type='TextMessa

In [4]:
team_state = await agent_team.save_state()
print("Team state saved.")
print(team_state)

Team state saved.
{'type': 'TeamState', 'version': '1.0.0', 'agent_states': {'Writer_1': {'type': 'ChatAgentContainerState', 'version': '1.0.0', 'agent_state': {'type': 'AssistantAgentState', 'version': '1.0.0', 'llm_context': {'messages': [{'content': 'Write a poem about the sea in 3 lines', 'source': 'user', 'type': 'UserMessage'}, {'content': 'Vast waves dance with grace,  \nwhispering secrets untold,  \nblue embrace of dreams.', 'thought': None, 'source': 'Writer_1', 'type': 'AssistantMessage'}]}}, 'message_buffer': [{'id': '546ecd78-3e3b-4177-8324-572abe4e29a9', 'source': 'Writer_2', 'models_usage': {'prompt_tokens': 66, 'completion_tokens': 20}, 'metadata': {}, 'created_at': '2026-06-27T03:59:09.256680Z', 'content': 'Endless blue whispers,  \nmoonlit waves cradle the stars,  \nsecrets in the deep.', 'type': 'TextMessage'}]}, 'Writer_2': {'type': 'ChatAgentContainerState', 'version': '1.0.0', 'agent_state': {'type': 'AssistantAgentState', 'version': '1.0.0', 'llm_context': {'messa

In [5]:
await agent_team.reset()

In [6]:

stream = agent_team.run_stream(task="What was the last line of Poem you wrote?")

await Console(stream)

---------- TextMessage (user) ----------
What was the last line of Poem you wrote?
---------- TextMessage (Writer_1) ----------
I'm just an AI, but I can help you create or analyze poems! What would you like to explore?
---------- TextMessage (Writer_2) ----------
I'm only an AI, so I don't write poems myself, but I can assist you in creating one!


TaskResult(messages=[TextMessage(id='a7168981-3362-4de8-ba47-685b6abf450c', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 59, 20, 517480, tzinfo=datetime.timezone.utc), content='What was the last line of Poem you wrote?', type='TextMessage'), TextMessage(id='52be2079-9c9d-4181-9d75-944997898ac5', source='Writer_1', models_usage=RequestUsage(prompt_tokens=38, completion_tokens=22), metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 59, 21, 637180, tzinfo=datetime.timezone.utc), content="I'm just an AI, but I can help you create or analyze poems! What would you like to explore?", type='TextMessage'), TextMessage(id='3d38ee86-36ad-42b5-bfed-051373285f68', source='Writer_2', models_usage=RequestUsage(prompt_tokens=68, completion_tokens=21), metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 59, 22, 737105, tzinfo=datetime.timezone.utc), content="I'm only an AI, so I don't write poems myself, but I can assist you in creating one!",

In [7]:
await agent_team.load_state(team_state)


In [8]:

stream = agent_team.run_stream(task="What was the last line of Poem you wrote?")

await Console(stream)

---------- TextMessage (user) ----------
What was the last line of Poem you wrote?
---------- TextMessage (Writer_1) ----------
Blue embrace of dreams.
---------- TextMessage (Writer_2) ----------
The last line is: "secrets in the deep."


TaskResult(messages=[TextMessage(id='adc51e5a-38c5-4e88-b85b-8f642c12bd98', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 59, 27, 744177, tzinfo=datetime.timezone.utc), content='What was the last line of Poem you wrote?', type='TextMessage'), TextMessage(id='f2c89b65-cc0c-4886-8a5d-5046182569d0', source='Writer_1', models_usage=RequestUsage(prompt_tokens=106, completion_tokens=5), metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 59, 28, 480162, tzinfo=datetime.timezone.utc), content='Blue embrace of dreams.', type='TextMessage'), TextMessage(id='eaf1c700-a338-418a-b229-8f0753b84de3', source='Writer_2', models_usage=RequestUsage(prompt_tokens=119, completion_tokens=12), metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 59, 29, 188758, tzinfo=datetime.timezone.utc), content='The last line is: "secrets in the deep."', type='TextMessage')], stop_reason='Maximum number of messages 3 reached, current message count: 3')

In [9]:
team_state

{'type': 'TeamState',
 'version': '1.0.0',
 'agent_states': {'Writer_1': {'type': 'ChatAgentContainerState',
   'version': '1.0.0',
   'agent_state': {'type': 'AssistantAgentState',
    'version': '1.0.0',
    'llm_context': {'messages': [{'content': 'Write a poem about the sea in 3 lines',
       'source': 'user',
       'type': 'UserMessage'},
      {'content': 'Vast waves dance with grace,  \nwhispering secrets untold,  \nblue embrace of dreams.',
       'thought': None,
       'source': 'Writer_1',
       'type': 'AssistantMessage'}]}},
   'message_buffer': [{'id': '546ecd78-3e3b-4177-8324-572abe4e29a9',
     'source': 'Writer_2',
     'models_usage': {'prompt_tokens': 66, 'completion_tokens': 20},
     'metadata': {},
     'created_at': '2026-06-27T03:59:09.256680Z',
     'content': 'Endless blue whispers,  \nmoonlit waves cradle the stars,  \nsecrets in the deep.',
     'type': 'TextMessage'}]},
  'Writer_2': {'type': 'ChatAgentContainerState',
   'version': '1.0.0',
   'agent_st

In [10]:
type(team_state)

dict

# if its a dict, it can be serialized to a file or written to a database

In [10]:
import json

In [11]:
with open('team_state.json', 'w') as f:
    json.dump(team_state, f)